In [2]:
from sklearn.cluster import HDBSCAN
from ForGetEmb.test import get_emb

/home/aidar/projects/mirea_projects/conf-bot/conf-bot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-02-25 16:41:36 nemo_logging:405] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


In [3]:
emb, meta, _, speech_segments_ms = get_emb()

[NeMo W 2026-02-25 16:41:38 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-02-25 16:41:38 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2026-02-25 16:41:38 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-25 16:41:38 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /home/aidar/projects/mirea_projects/conf-bot/conf-bot/data/models/titanet_large/speakerverification_en_titanet_large.nemo.


In [4]:
import numpy as np
import hdbscan

def cluster_hdbscan_cosine_precomputed(
    embeddings: np.ndarray,
    min_cluster_size: int = 20,
    min_samples: int | None = 10,
):
    """
    embeddings: (N, D), L2-normalized
    returns labels: (N,), with possible -1 (noise)
    """
    # cosine similarity (float64 for hdbscan)
    E = np.asarray(embeddings, dtype=np.float64)
    sim = E @ E.T
    sim = np.clip(sim, -1.0, 1.0)

    dist = 1.0 - sim
    dist = dist.astype(np.float64, copy=False)

    # clean up numerical issues
    np.fill_diagonal(dist, 0.0)
    dist = 0.5 * (dist + dist.T)

    clusterer = hdbscan.HDBSCAN(
        metric="precomputed",
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(dist)
    return labels

In [5]:
labels_hdb = cluster_hdbscan_cosine_precomputed(
    emb,
    min_cluster_size=20,
    min_samples=10,
)

num_clusters = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
noise_frac = float(np.mean(labels_hdb == -1))

print("HDBSCAN clusters:", num_clusters)
print("noise fraction:", noise_frac)
print("first 30 labels:", labels_hdb[:30])

HDBSCAN clusters: 2
noise fraction: 0.01485148514851485
first 30 labels: [ 0  0  0  0  0  0  1  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  1  1  1 -1  1]


In [9]:
segments_window_level = [(t0, t1, int(lbl)) for (t0, t1), lbl in zip(meta, labels_hdb)]


In [10]:
segments_window_level

[(1.328, 3.3280000000000003, 0),
 (1.828, 3.8280000000000003, 0),
 (2.3280000000000003, 4.328, 0),
 (2.8280000000000003, 4.828, 0),
 (3.3280000000000003, 5.328, 0),
 (3.8280000000000003, 5.828, 0),
 (6.944, 8.943999999999999, 1),
 (7.444, 9.443999999999999, 1),
 (7.944, 9.943999999999999, 1),
 (8.443999999999999, 10.443999999999999, 1),
 (8.943999999999999, 10.943999999999999, 1),
 (11.84, 13.84, 0),
 (12.34, 14.34, 0),
 (12.84, 14.84, 0),
 (13.34, 15.34, 0),
 (13.84, 15.84, 0),
 (16.608, 18.608, 0),
 (17.108, 19.108, 0),
 (17.608, 19.608, 0),
 (21.264, 23.264, 0),
 (21.764, 23.764, 0),
 (22.264, 24.264, 0),
 (22.764, 24.764, 0),
 (23.264, 25.264, 0),
 (26.752, 28.752, 0),
 (29.552, 31.552, 1),
 (30.052, 32.052, 1),
 (30.552, 32.552, 1),
 (31.052, 33.052, -1),
 (31.552, 33.552, 1),
 (32.052, 34.052, 1),
 (32.552, 34.552, 1),
 (33.052, 35.052, 1),
 (33.552, 35.552, 1),
 (34.052, 36.052, 1),
 (34.552, 36.552, 1),
 (37.504, 39.504, 1),
 (38.004, 40.004, 1),
 (38.504, 40.504, 1),
 (39.004,